<a href="https://colab.research.google.com/github/Fazal2204/IMLcodingquestions/blob/main/ImlCoding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Question 1 a)

In [ ]:
import numpy as np

class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None):
        self.feature=feature
        self.threshold=threshold
        self.left=left
        self.right=right
        self.value=value

class DecisionTree:
    def __init__(self,max_depth=10,min_samples_split=2,n_features=None):
        self.max_depth=max_depth
        self.min_samples_split=min_samples_split
        self.n_features=n_features
        self.root=None

    def build_tree(self,X,y,depth=0):
        if len(X)==0:
            return Node(value=0)
        n_samples=len(X)
        n_feats=len(X[0])
        if depth>=self.max_depth or n_samples<self.min_samples_split or len(set(y))==1:
            return Node(value=np.mean(y))
        feat_idxs=np.random.choice(n_feats,self.n_features,replace=False)
        best_feat,best_thresh=self._best_split(X,y,feat_idxs)
        if best_feat is None:
            return Node(value=np.mean(y))
        left_idxs,right_idxs=self._split(X,best_feat,best_thresh)
        if len(left_idxs)==0 or len(right_idxs)==0:
            return Node(value=np.mean(y))
        left=self.build_tree([X[i] for i in left_idxs],[y[i] for i in left_idxs],depth+1)
        right=self.build_tree([X[i] for i in right_idxs],[y[i] for i in right_idxs],depth+1)
        return Node(best_feat,best_thresh,left,right)

    def _best_split(self,X,y,feat_idxs):
        best_gain=-1
        split_idx=None
        split_thresh=None
        for feat_idx in feat_idxs:
            X_column=[row[feat_idx] for row in X]
            thresholds=set(X_column)
            for thr in thresholds:
                gain=self._variance_reduction(X,y,feat_idx,thr)
                if gain>best_gain:
                    best_gain=gain
                    split_idx=feat_idx
                    split_thresh=thr
        return split_idx,split_thresh

    def _variance_reduction(self,X,y,split_idx,threshold):
        parent_var=np.var(y)
        left_idxs,right_idxs=self._split(X,split_idx,threshold)
        if len(left_idxs)==0 or len(right_idxs)==0:
            return 0
        n=len(y)
        n_l=len(left_idxs)
        n_r=len(right_idxs)
        var_l=np.var([y[i] for i in left_idxs])
        var_r=np.var([y[i] for i in right_idxs])
        child_var=(n_l/n)*var_l+(n_r/n)*var_r
        return parent_var-child_var

    def _split(self,X,split_idx,threshold):
        left_idxs=[i for i,row in enumerate(X) if row[split_idx]<=threshold]
        right_idxs=[i for i,row in enumerate(X) if row[split_idx]>threshold]
        return left_idxs,right_idxs

    def fit(self,X,y):
        if self.n_features is None:
            self.n_features=X.shape[1]
        X_list=X.tolist()
        y_list=y.tolist()
        self.root=self.build_tree(X_list,y_list)

    def predict(self,x):
        node=self.root
        while node.value is None:
            if x[node.feature]<=node.threshold:
                node=node.left
            else:
                node=node.right
        return node.value

    def predict_batch(self,X):
        return np.array([self.predict(x) for x in X])

1b)

In [ ]:
class RandomForest:
    def __init__(self,n_trees=20,max_depth=10,min_samples_split=2,n_features=None):
        self.n_trees=n_trees
        self.max_depth=max_depth
        self.min_samples_split=min_samples_split
        self.n_features=n_features
        self.trees=[]

    def build_forest(self,X,y):
        self.trees=[]
        for _ in range(self.n_trees):
            tree=DecisionTree(max_depth=self.max_depth,
                              min_samples_split=self.min_samples_split,
                              n_features=self.n_features)
            X_samp,y_samp=self._bootstrap_samples(X,y)
            tree.fit(X_samp,y_samp)
            self.trees.append(tree)

    def _bootstrap_samples(self,X,y):
        n_samples=X.shape[0]
        idxs=np.random.choice(n_samples,n_samples,replace=True)
        return X[idxs],y[idxs]

    def predict(self,X):
        predictions=np.array([tree.predict_batch(X) for tree in self.trees])
        predictions=np.swapaxes(predictions,0,1)
        return np.array([np.mean(pred) for pred in predictions])

1c)

In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

X,y=load_diabetes(return_X_y=True)

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

single_tree=DecisionTree(max_depth=10)
single_tree.fit(X_train,y_train)
tree_preds=single_tree.predict_batch(X_test)
tree_mse=mean_squared_error(y_test,tree_preds)
print("Single Decision Tree MSE:",tree_mse)

rf=RandomForest(n_trees=20,max_depth=10,n_features=int(np.sqrt(X.shape[1])))
rf.build_forest(X_train,y_train)
rf_preds=rf.predict(X_test)
rf_mse=mean_squared_error(y_test,rf_preds)
print("Random Forest MSE:",rf_mse)

Single Decision Tree MSE: 4141.152167796453
Random Forest MSE: 3043.042157785543


Question 2 a)

In [ ]:
import numpy as np
import torchvision
import torchvision.transforms as transforms

transform=transforms.Compose([transforms.ToTensor()])

train_dataset=torchvision.datasets.FashionMNIST(root='./data',train=True,download=True,transform=transform)
test_dataset=torchvision.datasets.FashionMNIST(root='./data',train=False,download=True,transform=transform)

X_train=train_dataset.data.numpy().reshape(-1,28*28)/255.0
y_train=train_dataset.targets.numpy()

X_test=test_dataset.data.numpy().reshape(-1,28*28)/255.0
y_test=test_dataset.targets.numpy()

100%|██████████| 26.4M/26.4M [00:01<00:00, 14.4MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 263kB/s]
100%|██████████| 4.42M/4.42M [00:00<00:00, 4.93MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 8.86MB/s]


2 b)

In [ ]:
class LogisticRegressionScratch:
    def __init__(self,learning_rate=0.1,num_iterations=1000,num_classes=10):
        self.lr=learning_rate
        self.num_iterations=num_iterations
        self.num_classes=num_classes
        self.W=None
        self.b=None

    def _one_hot(self,y):
        m=y.shape[0]
        one_hot=np.zeros((m,self.num_classes))
        one_hot[np.arange(m),y]=1
        return one_hot

    def _softmax(self,z):
        z_shifted=z-np.max(z,axis=1,keepdims=True)
        exp_z=np.exp(z_shifted)
        return exp_z/np.sum(exp_z,axis=1,keepdims=True)

    def fit(self,X,y):
        m,n=X.shape
        self.W=np.zeros((n,self.num_classes))
        self.b=np.zeros(self.num_classes)
        y_hot=self._one_hot(y)

        for i in range(self.num_iterations):
            Z=np.dot(X,self.W)+self.b
            A=self._softmax(Z)
            dZ=A-y_hot
            dW=(1/m)*np.dot(X.T,dZ)
            db=(1/m)*np.sum(dZ,axis=0)
            self.W-=self.lr*dW
            self.b-=self.lr*db

            if i%100==0:
                loss=-np.mean(np.sum(y_hot*np.log(A+1e-8),axis=1))
                print(f"Iteration {i}, Loss: {loss:.4f}")

    def predict(self,X):
        Z=np.dot(X,self.W)+self.b
        A=self._softmax(Z)
        return np.argmax(A,axis=1)

2 c)

In [ ]:
model=LogisticRegressionScratch(learning_rate=0.05,num_iterations=500,num_classes=10)
model.fit(X_train,y_train)
predictions=model.predict(X_test)
accuracy=np.mean(predictions==y_test)
print(f"Test Set Accuracy: {accuracy*100:.2f}%")

Iteration 0, Loss: 2.3026
Iteration 100, Loss: 0.8304
Iteration 200, Loss: 0.7097
Iteration 300, Loss: 0.6530
Iteration 400, Loss: 0.6178
Test Set Accuracy: 79.94%


Question 3 a)

In [ ]:
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

X,y=load_diabetes(return_X_y=True)

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

linear_model=LinearRegression()
linear_model.fit(X_train,y_train)

y_pred_linear=linear_model.predict(X_test)
mse_linear=mean_squared_error(y_test,y_pred_linear)

print(f"Linear Regression MSE: {mse_linear:.4f}")

Linear Regression MSE: 2900.1936


3 b)

In [ ]:
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.kernel_ridge import KernelRidge
from sklearn.metrics import mean_squared_error

X,y=load_diabetes(return_X_y=True)

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

krr_poly=KernelRidge(kernel='poly',degree=2,alpha=1.0)
krr_poly.fit(X_train,y_train)
y_pred_poly=krr_poly.predict(X_test)
mse_poly=mean_squared_error(y_test,y_pred_poly)
print(f"Kernel Ridge Regression (Polynomial) MSE: {mse_poly:.4f}")

krr_rbf=KernelRidge(kernel='rbf',gamma=0.1,alpha=1.0)
krr_rbf.fit(X_train,y_train)
y_pred_rbf=krr_rbf.predict(X_test)
mse_rbf=mean_squared_error(y_test,y_pred_rbf)
print(f"Kernel Ridge Regression (RBF) MSE: {mse_rbf:.4f}")

Kernel Ridge Regression (Polynomial) MSE: 3969.7834
Kernel Ridge Regression (RBF) MSE: 3975.6521
